In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
import torch
import torch
import torchvision.models as models
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split


In [ ]:
from model import CRNNResNet

In [20]:
class avesDataset(Dataset):
        
        def __init__(self, root_dir):
            self.root_dir = Path(root_dir)
            self.samples = []
            self.label_map = {}

            for idx, class_dir in enumerate(sorted(self.root_dir.iterdir())):
                if not class_dir.is_dir():
                    continue

                self.label_map[class_dir.name] = idx

                for file in class_dir.glob("*.npy"):
                    self.samples.append((file, idx))

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            file_path, label = self.samples[idx]

            spec = np.load(file_path) 

            spec = (spec - spec.mean()) / (spec.std() + 1e-6)

            spec = torch.tensor(spec, dtype=torch.float32)
            spec = spec.unsqueeze(0) 
            return spec, label
        


In [21]:
dataset = avesDataset(root_dir="../dataset")

In [22]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [23]:
data_dir = "../dataset"
classes = sorted([p.name for p in Path(data_dir).iterdir() if p.is_dir()])

classToIdx = {cls: i for i, cls in enumerate(classes)}
targetClass = {i: cls for cls, i in classToIdx.items()}

targetClass

{0: 'aracuã-de-barriga-branca',
 1: 'aracuã-do-pantanal',
 2: 'aracuã-escamoso',
 3: 'jacuguaçu',
 4: 'jacupemba',
 5: 'mutum-de-penacho'}

Modelo

In [24]:
def train_model(num_epochs, batch_size, train_loader, val_loader, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Usando dispositivo: {device}")

    # Inicializar modelo, perda e otimizador
    model = CRNNResNet(num_classes=6).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Loop de treinamento
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_acc = 100. * correct / total
        
        # Validação em cada época
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_acc = 100. * val_correct / val_total
        print(f"Época [{epoch+1}/{num_epochs}] - "
              f"Loss Treino: {running_loss/len(train_loader):.4f}, Acc Treino: {train_acc:.2f}% | "
              f"Loss Val: {val_loss/len(val_loader):.4f}, Acc Val: {val_acc:.2f}%")


In [25]:
def evaluate_model(model, loader, criterion, device, phase_name="Teste"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    accuracy = 100. * correct / total
    avg_loss = running_loss / len(loader)
    print(f"\n--- Resultado {phase_name} ---")
    print(f"Loss: {avg_loss:.4f} | Acurácia: {accuracy:.2f}%")
    return accuracy


In [26]:
# Split Treino (80%), Validação (10%), Teste (10%)
full_dataset = avesDataset(data_dir)

total_len = len(full_dataset)
train_size = int(0.8 * total_len)
val_size = int(0.1 * total_len)
test_size = total_len - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
train_model(num_epochs=100, batch_size=16, train_loader=train_loader, val_loader=val_loader)
evaluate_model(loader=test_loader, model = "aves_crnn_resnet.pth")

Usando dispositivo: cuda


c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Época [1/100] - Loss Treino: 1.6952, Acc Treino: 27.79% | Loss Val: 2.0174, Acc Val: 22.06%
Época [2/100] - Loss Treino: 1.5837, Acc Treino: 29.98% | Loss Val: 2.1792, Acc Val: 32.35%
Época [3/100] - Loss Treino: 1.4463, Acc Treino: 36.38% | Loss Val: 1.2505, Acc Val: 41.18%
Época [4/100] - Loss Treino: 1.4091, Acc Treino: 37.66% | Loss Val: 2.2040, Acc Val: 26.47%
Época [5/100] - Loss Treino: 1.3378, Acc Treino: 39.31% | Loss Val: 1.5923, Acc Val: 27.94%
Época [6/100] - Loss Treino: 1.4346, Acc Treino: 35.83% | Loss Val: 1.9005, Acc Val: 26.47%
Época [7/100] - Loss Treino: 1.4349, Acc Treino: 37.66% | Loss Val: 1.3098, Acc Val: 39.71%
Época [8/100] - Loss Treino: 1.2810, Acc Treino: 39.12% | Loss Val: 1.5127, Acc Val: 32.35%
Época [9/100] - Loss Treino: 1.2323, Acc Treino: 41.13% | Loss Val: 1.3990, Acc Val: 45.59%
Época [10/100] - Loss Treino: 1.3018, Acc Treino: 41.86% | Loss Val: 1.8719, Acc Val: 32.35%
Época [11/100] - Loss Treino: 1.2367, Acc Treino: 42.05% | Loss Val: 1.3524, Ac

TypeError: evaluate_model() missing 3 required positional arguments: 'model', 'criterion', and 'device'

In [29]:

# ===== device =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== recria o modelo =====
# ⚠️ use exatamente a mesma classe/arquitetura do treino
model = CRNNResNet(
    num_classes=6  # mesmo valor usado no treino
)

model = model.to(device)

# ===== carrega os pesos =====
checkpoint_path = "aves_crnn_resnet.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

print("Modelo carregado com sucesso!")

# ===== loss =====
criterion = nn.CrossEntropyLoss()

# ===== avalia =====
accuracy = evaluate_model(
    model=model,
    loader=test_loader,     # DataLoader de teste
    criterion=criterion,
    device=device,
    phase_name="Teste"
)

c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Modelo carregado com sucesso!

--- Resultado Teste ---
Loss: 1.0093 | Acurácia: 52.17%
